# FCIS 2026 Scene Classification — Colab GPU training

This notebook trains five models on a Colab GPU: four course models (including a from-scratch ViT transformer) plus a pretrained ResNet18 bonus model. It downloads the competition data using your Kaggle API credentials, compares models on the same stratified validation split, and creates test predictions.

Before running: set Colab runtime to **T4 GPU** (Runtime → Change runtime type), accept the Kaggle competition rules, and add `KAGGLE_API_TOKEN` in the Colab Secrets panel. Enable notebook access to it. The notebook clones this public GitHub repository and downloads the competition data directly from Kaggle; no dataset files or API tokens are stored in GitHub.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, zipfile, subprocess, sys, json, csv

assert __import__('torch').cuda.is_available(), 'Enable a GPU runtime first: Runtime → Change runtime type → T4 GPU'
print(__import__('torch').cuda.get_device_name(0))
drive.mount('/content/drive')
project = Path('/content/fcis-scene-classification')
if not project.exists(): subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dandan29376060887/fcis-scene-classification.git', str(project)], check=True)
os.chdir(project)
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
DATA_ROOT = Path('/content/Data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
archive = Path('/content/fcis-2026-nn-top-view-scene-classification.zip')
if not archive.exists():
    subprocess.run(['kaggle', 'competitions', 'download', '-c', 'fcis-2026-nn-top-view-scene-classification', '-p', '/content'], check=True)
with zipfile.ZipFile(archive) as z: z.extractall(DATA_ROOT)
# Kaggle competitions sometimes package train/test files in ZIPs inside the download.
for _ in range(5):
    nested_archives = list(DATA_ROOT.rglob('*.zip'))
    if not nested_archives: break
    for nested in nested_archives:
        destination = nested.with_suffix('')
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(nested) as z: z.extractall(destination)
label_files = list(DATA_ROOT.rglob('labels.csv'))
if not label_files:
    raise FileNotFoundError(f'No labels.csv under {DATA_ROOT}; extracted files: {[p.name for p in DATA_ROOT.iterdir()]}')
data_root = next((p.parent for p in label_files if (p.parent / 'test.csv').exists()), label_files[0].parent)
image_dirs = [p for p in data_root.rglob('*') if p.is_dir() and any(p.glob('*.jpg'))]
train_dirs = [p for p in image_dirs if p.name.lower() in ('data', 'train', 'training')]
test_dirs = [p for p in image_dirs if p.name.lower() in ('test', 'testing')]
if not train_dirs or not test_dirs:
    raise FileNotFoundError(f'Could not locate train/test JPG folders under {data_root}; found: {[str(p) for p in image_dirs[:20]]}')
train_image_dir = max(train_dirs, key=lambda p: len(list(p.glob('*.jpg'))))
test_image_dir = max(test_dirs, key=lambda p: len(list(p.glob('*.jpg'))))
train_count, test_count = len(list(train_image_dir.glob('*.jpg'))), len(list(test_image_dir.glob('*.jpg')))
with open(data_root / 'labels.csv', newline='') as f: label_count = sum(1 for _ in csv.DictReader(f))
with open(data_root / 'test.csv', newline='') as f: test_id_count = sum(1 for _ in csv.DictReader(f))
assert train_count and test_count and label_count and test_id_count, 'Dataset folders or CSVs are empty.'
assert train_count >= label_count and test_count >= test_id_count, 'Some image files are missing.'
print(f'Dataset ready: root={data_root} | train JPGs={train_count}, labels={label_count} | test JPGs={test_count}, test IDs={test_id_count}')


In [ ]:
# Train all four scratch models (ViT satisfies the transformer requirement) and a pretrained bonus model.
models = ['simple_cnn', 'vgg_style', 'residual_cnn', 'se_cnn', 'vit_scratch', 'resnet18_transfer']
results = []
for name in models:
    cfg = json.loads(Path(f'configs/{name if name != "resnet18_transfer" else "residual_cnn"}.json').read_text())
    cfg['data'].update(root=str(data_root), train_dir=train_image_dir.relative_to(data_root).as_posix(), test_dir=test_image_dir.relative_to(data_root).as_posix(), batch_size=64, num_workers=2, image_size=224, seed=42, val_fraction=0.15)
    cfg['model']['name'] = name
    cfg['model']['pretrained'] = (name == 'resnet18_transfer')
    if name == 'vit_scratch':
        cfg['model']['extra'] = {'patch_size':16, 'depth':4, 'heads':4, 'mlp_ratio':4.0, 'image_size':224}
    cfg['train'].update(epochs=40, early_stopping_patience=8, lr=(3e-4 if name == 'vit_scratch' else 1e-3), out_dir=f'/content/drive/MyDrive/fcis_runs/{name}', seed=42)
    Path(f'/content/{name}.json').write_text(json.dumps(cfg, indent=2))
    print(f'\n===== Training {name} =====', flush=True)
    subprocess.run([sys.executable, 'scripts/train.py', '--config', f'/content/{name}.json'], check=True)
    history = json.loads(Path(cfg['train']['out_dir'], 'history.json').read_text())
    best = max(history, key=lambda row: row['val_acc'])
    results.append({'model': name, 'best_val_acc': best['val_acc'], 'best_epoch': best['epoch'], 'run_dir': cfg['train']['out_dir']})
    print(results[-1])
Path('/content/drive/MyDrive/fcis_runs/model_comparison.json').write_text(json.dumps(results, indent=2))
sorted(results, key=lambda x: x['best_val_acc'], reverse=True)

In [ ]:
# Create a Kaggle-format submission from the model with the best held-out validation accuracy.
winner = max(results, key=lambda x: x['best_val_acc'])
print('Selected on held-out validation:', winner['model'], winner['best_val_acc'])
winner_cfg = f"/content/{winner['model']}.json"
out_csv = '/content/drive/MyDrive/fcis_runs/submission.csv'
subprocess.run([sys.executable, 'scripts/predict.py', '--config', winner_cfg, '--checkpoint', f"{winner['run_dir']}/best.pt", '--data-root', str(data_root), '--out', out_csv, '--tta'], check=True)
import pandas as pd
submission = pd.read_csv(out_csv)
assert list(submission.columns) == ['image_id', 'label'] and len(submission) == 1100
assert submission['label'].notna().all()
print('Saved:', out_csv, '| rows:', len(submission))
files.download(out_csv)